In [3]:
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import joblib

# 1. Carregar os dados do seu arquivo JSON do Firebase
with open('consumo-conciente.json', 'r') as f:
    dados_firebase = json.load(f)

# Pegar os dados que estão dentro do nó principal "leituras"
if "leituras" in dados_firebase:
    dados_raiz = dados_firebase["leituras"]
else:
    dados_raiz = dados_firebase

# Lista para armazenar as linhas achatadas
linhas_achatadas = []

# Varrer a árvore do JSON: Data -> Hora -> Atributos Elétricos
for data, horas in dados_raiz.items():
    if isinstance(horas, dict):
        for hora, atributos in horas.items():
            # FILTRO CRÍTICO: Garante que estamos a ler o dicionário da leitura e ignora metadados como 'consumo_referencia_dia'
            if isinstance(atributos, dict):
                registro = atributos.copy()

                # Sistema de tolerância (fallbacks) caso existam chaves com nomenclaturas antigas
                if 'consumo' in registro and 'consumo_total_pzem' not in registro:
                    registro['consumo_total_pzem'] = registro['consumo']
                elif 'consumo_acumulado' in registro and 'consumo_total_pzem' not in registro:
                    registro['consumo_total_pzem'] = registro['consumo_acumulado']

                # Armazena os dados elétricos estruturados
                linhas_achatadas.append(registro)

# Criar o DataFrame com as linhas planas
df = pd.DataFrame(linhas_achatadas)

# --- MUDANÇA DATASET NOVO ---
# Definir as colunas monitoradas usando os nomes exatos do novo JSON
COLUNAS_MONITORADAS = ['consumo_total_pzem', 'corrente', 'potencia', 'voltagem']

# Validação: Garante que todas as colunas exigidas existem no DataFrame para evitar KeyError
for col in COLUNAS_MONITORADAS:
    if col not in df.columns:
        df[col] = 0.0

df_filtrado = df[COLUNAS_MONITORADAS].dropna().copy()

print(f"Total de amostras prontas para o treino: {len(df_filtrado)}")
print("Exemplo dos dados mapeados:\n", df_filtrado.head())

# 2. Normalização dos dados (0 a 1)
scaler = MinMaxScaler(feature_range=(0, 1))
dados_normalizados = scaler.fit_transform(df_filtrado)

# Salvar o scaler elétrico atualizado
joblib.dump(scaler, 'scaler_energia.pkl')
print("O arquivo 'scaler_energia.pkl' foi gerado com sucesso!")

# 3. Criação da Janela Temporal (60 passos)
X, y = [], []
JANELA = 60
INDICE_CONSUMO = COLUNAS_MONITORADAS.index('consumo_total_pzem')

for i in range(JANELA, len(dados_normalizados)):
    X.append(dados_normalizados[i-JANELA:i])       # As 60 leituras anteriores das 4 features
    y.append(dados_normalizados[i, INDICE_CONSUMO]) # O consumo_total_pzem do próximo passo (Target)

X, y = np.array(X), np.array(y)

if len(X) == 0:
    raise ValueError(f"Dados insuficientes para criar a janela temporal. Tem apenas {len(dados_normalizados)} amostras, o mínimo necessário são {JANELA + 1}.")

# 4. Construção e Treinamento do Modelo LSTM
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1) # Saída: Consumo Previsto (consumo_total_pzem)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, batch_size=32, epochs=20)

# Salvar os arquivos finais da IA
model.save('modelo_lstm_energia.keras')
print("\n" + "="*50)
print("Modelo treinado focado em CONSUMO TOTAL PZEM salvo com sucesso!")
print("Arquivos prontos: 'modelo_lstm_energia.keras' e 'scaler_energia.pkl'")
print("="*50)

Total de amostras prontas para o treino: 3777
Exemplo dos dados mapeados:
    consumo_total_pzem  corrente  potencia  voltagem
0               0.167     0.132       9.5     126.9
1               0.169     0.208      16.0     126.6
2               0.172     0.211      16.0     127.0
3               0.175     0.210      16.0     126.4
4               0.177     0.110       7.7     127.7
O arquivo 'scaler_energia.pkl' foi gerado com sucesso!


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - loss: 0.0149
Epoch 2/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0027
Epoch 3/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - loss: 0.0023
Epoch 4/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0018
Epoch 5/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - loss: 0.0016
Epoch 6/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - loss: 0.0015
Epoch 7/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0013
Epoch 8/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - loss: 0.0011
Epoch 9/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - loss: 9.7123e-04
Epoch 10/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - loss: 9.0764e-04
Epoch 11/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 9.1449e-04
Epoch 12/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - loss: 7.6159e-04
Epoch 13/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - loss: 7.3753e-04
Epoch 14/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - loss: 6.4168e-04
Epoch 15/20
117

In [ ]:
from google.colab import drive
drive.mount('/content/drive')